# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

##### 1 
* a / 2 — обычное деление, результат всегда дробный: [1. 2. 3.].
* a // 4 — целочисленное деление без остатка: 2//4 = 0, 4//4 = 1, 6//4 = 1, получается [0 1 1].

##### 2
* np.ones((3, 4)) — матрица 3*4 из единиц (дробных).
* axis=1 складывает каждую строку: по 4 единицы, итого 4. Строк 3, поэтому 3 числа.

##### 3
* np.arange(3) — строка [0 1 2] формы (3,).
* .reshape(3, 1) — столбец [[0],[1],[2]] формы (3, 1).
* Сработает broadcasting: строка растягивается вниз, столбец вправо, и складывается каждая пара. Получается таблица сложения 3*3.

##### 4
* np.sqrt([1, 4, 9]) даёт [1. 2. 3.], сумма равна 6.0.

##### 5 
* np.arange(1, 5) даёт [1 2 3 4] (5 не включается).
* ** 2 даёт [1 4 9 16].
* Среднее: (1 + 4 + 9 + 16) / 4 = 30 / 4 = 7.5.

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [3]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [4]:
# 1
sept = amount[clean & (month=='2026-09')].sum()
october = amount[clean & (month=='2026-10')].sum()

changes = (october-sept)/sept
changes
print(f"{'Выручка за сентябрь:':>22} {sept:>14,.0f} ₸")
print(f"{'Выручка за октябрь:':>22} {october:>14,.0f} ₸")
print(f"{'Изменение:':>22} {changes:>+14.2%}")

  Выручка за сентябрь:     35,243,410 ₸
   Выручка за октябрь:     34,419,520 ₸
            Изменение:         -2.34%


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [5]:
# 2
category = np.unique(cat)
months = np.unique(month)
category
table = np.array([[amount[clean & (cat == c) & (month == m)].sum() for m in months] for c in category])
for c, r in zip(category, table):
    print(f"{c:<10} {r[0]:>12,.0f} {r[1]:>12,.0f}")
print(table.shape)

Дом           4,997,730    5,125,790
Книги         1,065,300    1,140,910
Красота       2,610,950    2,678,990
Одежда        9,167,100    8,505,730
Электроника   17,402,330   16,968,100
(5, 2)


In [6]:
print(f"{'Категория, принёсшая больше всего выручки за два месяца: '}{category[table.sum(axis=1).argmax()]}")

Категория, принёсшая больше всего выручки за два месяца: Электроника


In [7]:
changes_category = table[:, 1] / table[:, 0] - 1
print(f"{'Категория, упавшая сильнее всех:'} {category[changes_category.argmin()]}")
print(f"{'Процент спада:'} {changes_category.min():.2%}")

Категория, упавшая сильнее всех: Одежда
Процент спада: -7.21%


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [8]:
# 3
table_new = np.array([[(clean & (cat == c) & (month == m)).sum() for m in months]
                   for c in category])
for c, r in zip(category, table_new):
    print(f"{c:<10} {r[0]:>12,.0f} {r[1]:>12,.0f}")
print(table.shape)

Дом                 476          476
Книги               265          299
Красота             422          435
Одежда              678          652
Электроника          531          505
(5, 2)


In [9]:
avg = table/table_new
for c, r in zip(category, avg):
    print(f"{c:<10} {r[0]:>12,.0f} {r[1]:>12,.0f}")
print(table.shape)

Дом              10,499       10,768
Книги             4,020        3,816
Красота           6,187        6,159
Одежда           13,521       13,046
Электроника       32,773       33,600
(5, 2)


In [10]:
print(f"{'Категория с наибольшим средним чеком: '}{category[avg.sum(axis=1).argmax()]}")

Категория с наибольшим средним чеком: Электроника


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [11]:
# 4
cities = np.unique(city)
months = np.array(["2026-09", "2026-10"])

# Таблица 6 × 2: строка = город, столбец = месяц
table_city = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months]
                  for c in cities])
for c, r in zip(cities, table_city):
    print(f"{c:<10} {r[0]:>12,.0f} {r[1]:>12,.0f}")
print(table_city.shape)

Актобе        2,576,960    2,540,420
Алматы       14,501,070   14,643,020
Астана        9,034,220    8,850,530
Атырау          923,990      969,860
Караганда     3,648,520    3,371,680
Шымкент       4,558,650    4,044,010
(6, 2)


In [12]:
table_city.sum(axis=0)

array([35243410., 34419520.])

In [25]:
x = table_city / table_city.sum(axis=0)
for c, r in zip(cities, x):
    print(f"{c:<10} {r[0]:>12,.3f} {r[1]:>12,.3f}")
print(x.shape)

Актобе            0.073        0.074
Алматы            0.411        0.425
Астана            0.256        0.257
Атырау            0.026        0.028
Караганда         0.104        0.098
Шымкент           0.129        0.117
(6, 2)


Доля Шымкента от сентября к октябрю сильно упала 

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [35]:
# 5
city_change = table_city[:, 1] - table_city[:, 0]   
total_change = city_change.sum()                    
contribution = city_change / total_change * 100      

order = np.argsort(contribution)[::-1]           
print(f"{'Город':<10} {'Изменение, ₸':>14} {'Вклад':>7}")
for i in order:
    print(f"{cities[i]:<10} {city_change[i]:>+14,.0f} {contribution[i]:>6.1f}%")

Город        Изменение, ₸   Вклад
Шымкент          -514,640   62.5%
Караганда        -276,840   33.6%
Астана           -183,690   22.3%
Актобе            -36,540    4.4%
Атырау            +45,870   -5.6%
Алматы           +141,950  -17.2%


Причину падения ищите в Шымкенте: он один даёт 62% снижения выручки.

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [41]:
# 6
share = table / table.sum(axis=1, keepdims=True)
share

array([[0.49367512, 0.50632488],
       [0.48286428, 0.51713572],
       [0.49356893, 0.50643107],
       [0.51871149, 0.48128851],
       [0.50631691, 0.49368309]])

table имеет форму (6, 2), а table.sum(axis=1) имеет форму (6,). NumPy сопоставляет формы с правого края: 2 столбца против 6 значений не совпадают, поэтому деление невозможно (ValueError: operands could not be broadcast together). Сумму по строкам нужно превратить в столбец формы (6, 1), тогда каждая строка поделится на свою сумму.
Без keepdims=True сумма по строкам «теряет» одно измерение и становится плоским списком поэтому добавляем keepdims=True

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8